## 3.2 Statistical Relationships Examined: Books

**Scope.** This section analyses **only reviews in the `Books` category** (47,179 reviews after D2 cleaning). Reviews of all other products are removed before any test is run, and no comparison with other categories is made.

This section examines three statistical relationships using `scipy`:

| Part | Method | Question (Books reviews only) |
|---|---|---|
| 1 | Welch's t-test | Among book reviews rated 4 or 5 stars, is the average star rating of reviews that contain a negative word (such as “bad” or “hate”) different from the average of reviews that do not? |
| 2 | Spearman correlation | Is the number of helpful votes on a book review related to its length? |
| 3 | Simple linear regression | How well does review length predict the number of helpful votes on a book review? |

**A note on p-values.** With 47,179 book reviews, even very small differences are "statistically significant". For every result we therefore also report an effect size, which measures how strong the relationship is. When Python prints `p = 0`, the value is too small to represent, and we report it as p < 0.001.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- 3.2 Setup -----------------------------------------------------------
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42
np.random.seed(SEED)

# If this section runs in the same notebook as D2, reuse the cleaned DataFrame:
#     rel_all = df_webDisplay.copy()
# Otherwise, load the file exported by D2 (319,357 rows, all categories):
DATA_PATH = "/content/drive/MyDrive/Introduction_of_Data_Science_Research/Dataset/2.1_amazon_reviews_webDisplay.csv"
rel_all = pd.read_csv(DATA_PATH)

# Keep Books only. Every test below uses this subset and nothing else.
rel = rel_all[rel_all["product_category"] == "Books"].copy()
assert len(rel) > 0, "No 'Books' rows found - check the spelling of the category name."

ALPHA = 0.05   # significance level, fixed before looking at any results
print(f"Books reviews: {len(rel):,} rows, {rel.shape[1]} columns")

def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

Books reviews: 47,179 rows, 19 columns


In [ ]:
# --- Part 1. T-test: star rating of high-rated book reviews with vs without a negative keyword ---
# Group A: 4-5 star book reviews whose text contains a negative keyword (D2 flag = 1)
# Group B: 4-5 star book reviews whose text does not (D2 flag = 0)
# H0: both groups have the same mean star rating.
high = rel[rel["star_rating"] >= 4]
a = high.loc[high["high_rating_negative_flag"] == 1, "star_rating"]
b = high.loc[high["high_rating_negative_flag"] == 0, "star_rating"]

# Welch's t-test: compares two means without assuming equal variances.
t, p = stats.ttest_ind(a, b, equal_var=False)

# Cohen's d: difference in means in standard-deviation units (0.2 small, 0.5 medium, 0.8 large).
pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
d = (a.mean() - b.mean()) / pooled_sd
diff = a.mean() - b.mean()

print(f"with negative word   : n = {len(a):,}, mean rating = {a.mean():.3f}")
print(f"without negative word: n = {len(b):,}, mean rating = {b.mean():.3f}")
print(f"difference = {diff:.3f} stars, t = {t:.1f}, {fmt_p(p)}, Cohen's d = {d:.3f}")
print("Reject H0" if p < ALPHA else "Fail to reject H0")

with negative word   : n = 3,818, mean rating = 4.749
without negative word: n = 23,937, mean rating = 4.826
difference = -0.078 stars, t = -10.4, p < 0.001, Cohen's d = -0.200
Reject H0


**Part 1 result.** Among 4-5 star book reviews, the 3,818 reviews containing a negative keyword have a mean rating of 4.749, compared with 4.826 for the 23,937 reviews that do not. The difference is 0.078 stars, and the t-test rejects the null hypothesis that the two means are equal (t = -10.4, p < 0.001).

Because every review in this group has either 4 or 5 stars, the mean is easy to read: a mean of 4.749 means that about 75% of the flagged reviews give 5 stars, compared with about 83% of the reviews without a negative keyword. Readers who use a negative word are therefore slightly more likely to give 4 stars instead of 5.

The difference is real but small: Cohen's d is -0.20, exactly at the conventional threshold for a small effect. In practical terms, a high-rated book review that contains a negative word receives almost the same star rating as one that does not. The star rating cannot separate the two groups, so whatever the negative word expresses is visible only in the review text.

In [4]:
# --- Part 2. Correlation: helpful votes vs review length (Books) ----------
# helpful_votes is highly skewed, so we use Spearman's correlation,
# which is rank-based and robust to a few extreme reviews.
print(rel[["review_length", "helpful_votes"]].describe().round(1))

rho, p_rho = stats.spearmanr(rel["review_length"], rel["helpful_votes"])
print(f"\nSpearman correlation = {rho:.3f}, {fmt_p(p_rho)}")

       review_length  helpful_votes
count        47179.0        47179.0
mean          1156.7           19.1
std           1719.0          168.8
min              2.0            3.0
25%            316.0            3.0
50%            646.0            5.0
75%           1315.0           11.0
max          43840.0        27550.0

Spearman correlation = 0.267, p < 0.001


**Part 2 result.** The Spearman correlation between review length and helpful votes for book reviews is 0.267 (p < 0.001). The correlation is positive, meaning longer book reviews tend to receive more helpful votes, but it is weak (below 0.3). Many short reviews still receive many votes and many long reviews receive few, so review length on its own tells us only a little about how helpful readers find a book review.

The summary table above shows why we use Spearman's rank correlation: helpful votes have a median of 5 but a maximum of 27,550, so a few extreme reviews would dominate an ordinary (Pearson) correlation.

In [5]:
# --- Part 3. Simple linear regression: review length -> helpful votes (Books) ---
# Both variables are right-skewed, so we fit the line on their logarithms:
#     log(helpful_votes) = intercept + slope * log(review_length)
# (D2 kept only reviews with >= 3 helpful votes; rows with length 0 are dropped so the log is defined.)
reg = rel[(rel["review_length"] > 0) & (rel["helpful_votes"] > 0)]
fit = stats.linregress(np.log(reg["review_length"]), np.log(reg["helpful_votes"]))
r2 = fit.rvalue ** 2
pct = 100 * (2 ** fit.slope - 1)

print(f"n = {len(reg):,}")
print(f"slope = {fit.slope:.3f}, intercept = {fit.intercept:.3f}")
print(f"R squared = {r2:.3f}, {fmt_p(fit.pvalue)}")
print(f"a review twice as long gets about {pct:.1f}% {'more' if pct >= 0 else 'fewer'} helpful votes on average")

n = 47,179
slope = 0.269, intercept = 0.241
R squared = 0.088, p < 0.001
a review twice as long gets about 20.5% more helpful votes on average


**Part 3 result.** The fitted line has a slope of 0.269 on the log scale. Because both variables are on a log scale, the slope can be read as a percentage: a book review twice as long receives about 20.5% more helpful votes on average (p < 0.001).

R² is 0.088, so review length explains only about 8.8% of the variation in helpful votes on book reviews. The other 91% depends on factors that are not in this model. The regression confirms the relationship found in Part 2 and shows how weak it is: longer book reviews do get somewhat more helpful votes, but review length is a poor predictor of how helpful a particular book review will be.

In [7]:
# --- Random sample of 10 flagged book reviews, full text -----------------
pd.set_option("display.max_colwidth", None)          # no character limit
sample = high[high["high_rating_negative_flag"] == 1].sample(10, random_state=SEED)

full = sample[["star_rating", "review_body"]].copy()
full["review_body"] = full["review_body"].str.replace("<br />", "\n", regex=False)

(full.style
     .set_properties(subset=["review_body"], **{"white-space": "pre-wrap", "text-align": "left", "min-width": "700px"})
     .set_properties(subset=["star_rating"], **{"text-align": "center"})
     .format({"star_rating": "{:.0f}"}))

**What the sample shows.** We read the 10 flagged book reviews above in full and judged whether the negative wording reflects a genuine complaint about the book (for example its content, writing, editing, printing or price), or whether it appears for another reason.

| Review ID | Stars | Negative wording in context | Genuine complaint? |
|---|---|---|---|
| 223734 | 4 | "bad things", "bad genes", "worst offenders"; also "This is not a perfect book", "skim the first 400 or so pages" | Yes, partly: the keywords describe the book's topic, but the review criticises the book's structure and argument |
| 273587 | 5 | "really good (bad) humor"; "this book is basic" | No, playful praise; "basic" is advice on who the book suits |
| 259161 | 5 | "Don't be boring", "The worst presentations" | No, summarises and quotes the book's advice |
| 307133 | 5 | "Rich Dad, Poor Dad", "the middle class and poor" | No, the title of another book and the book's topic |
| 291122 | 5 | "a couple of bad reviews here" | No, disagrees with other reviewers |
| 201837 | 5 | "getting a bad rap", "I hated Kakutani's review" | No, defends the book against a critic |
| 292593 | 5 | "completely wrong about him", "worse-off", "annoying" | No, about the reviewer's first impression and a different author |
| 296297 | 5 | "exaggerated, made up", "ripping each other to shreds" | No, dismisses the controversy and defends the book |
| 271160 | 5 | "eating meat is cruel", "true horror and evil" | No, describes the book's subject (the food industry) |
| 279719 | 5 | "never a dull word", "a poor farm kid", "bad guys" | No, negated, and describes the story |

**Count: 1 of 10 is a genuine complaint; 9 are not.** The only review that really criticises the book is also the only 4-star review in the sample, which is consistent with Part 1 (flagged reviews give 4 stars slightly more often). Ten reviews are far too few to draw a conclusion, but the pattern is clear: in book reviews, negative words are mostly used to describe the book's subject, to disagree with other reviewers or critics, or in a negated or playful way. A flag should therefore be read as "contains a negative word", not as "reports a problem with the book".

### Interpretation

Two findings are relevant to the decision question for **Books**.

First, about one in seven high-rated book reviews (3,818 of 27,755, or 13.8%) contains a negative keyword, yet their star rating is almost identical to that of the other reviews (4.75 vs 4.83, a small effect with Cohen's d = -0.20). A dashboard based on average star ratings cannot tell these book reviews apart.

Second, what readers find helpful in a book review is only weakly related to a simple measure such as review length (Spearman correlation 0.27, R² = 0.088). Doubling the length of a review adds about 20% more helpful votes on average, but length explains less than a tenth of the variation, so the useful content of a book review cannot be summarised by a single number either.

Both findings support analysing book review text alongside star ratings. They do not yet show that the text reveals problems with the books: in the sample we read, only 1 of 10 flagged reviews contained a genuine complaint about the book.

### What This Analysis Does Not Establish

- **These are associations, not causal effects.** We found patterns, not causes. We have not shown that negative wording lowers a book's star rating, or that writing a longer review earns more helpful votes.
- **A flag does not always mean a complaint about the book..** We read 10 randomly selected flagged book reviews (table above), and only 1 was a genuine complaint. The other 9 were flagged because the review described the book's subject, disagreed with other reviewers or a critic, criticised a different author, or used a negative word in a negated or playful way. So the real number of complaints is much lower than 3,818, but 10 reviews are too few to say how much lower.
- **Statistical significance here mainly reflects sample size.** With 47,179 book reviews, even tiny differences are significant. What matters is the effect size, and all three effect sizes are small or weak.
- **The t-test is a rough tool in this setting.** Within the 4-5 star group, the star rating takes only two values, so the data are far from normally distributed. With samples this large the test remains valid, but it essentially restates the difference between two proportions (about 75% vs 83% five-star reviews).
- **The sample is filtered.** D2 kept only verified purchases with at least 3 helpful votes. Book reviews with 0 to 2 votes are excluded, so the true relationship between review length and helpfulness across all book reviews may differ from what we measured.
- **Some review texts contain duplicated passages.** Two of the ten reviews we read (273587 and 223734) repeat whole paragraphs, which looks like a problem in the source data. This inflates `review_length` for such reviews and adds noise to Parts 2 and 3.
- **The results apply to Books only.** They should not be read as describing any other product category.
- **It says nothing about returns or sales.** The dataset contains no return or revenue variable, so we cannot test whether acting on book review text would reduce return rates.